In [ ]:
import os
import re
import warnings
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", message=".*cache-system uses symlinks.*")

DATA_PATH = "/content/Appliances.json"
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"
RANDOM_STATE = 42
MIN_WORDS_LONG_REVIEW = 100
TARGET_SUMMARY_WORDS = 50
MAX_INPUT_CHARS = 2200
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()

def word_count(text: str) -> int:
    return len(normalize_space(text).split())

def truncate_for_model(text: str, max_chars: int = MAX_INPUT_CHARS) -> str:
    text = normalize_space(text)
    if len(text) <= max_chars:
        return text
    cut = text[:max_chars]
    last_space = cut.rfind(" ")
    if last_space > 0:
        cut = cut[:last_space]
    return cut.strip()

def cap_to_approx_words(text: str, max_words: int = TARGET_SUMMARY_WORDS) -> str:
    words = normalize_space(text).split()
    if len(words) <= max_words:
        return " ".join(words)
    return " ".join(words[:max_words]).rstrip(" ,;:-") + "..."

def is_question_like(text: str) -> bool:
    t = normalize_space(text)
    if "?" in t: return True
    t2 = re.sub(r"[^a-zA-Z\s]", " ", t.lower())
    starters = ("how", "what", "why", "when", "where", "which", "does", "do", "did", "is", "are", "can", "could")
    return any(normalize_space(t2).startswith(s + " ") for s in starters)

class LocalMistralGenerator:
    def __init__(self, model_name: str):
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
            device_map="auto"
        )
        self.model.eval()

    def generate(self, prompt: str, max_new_tokens: int = 100):
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):].strip()

print("Loading dataset...")

df = pd.read_json(DATA_PATH, lines=True)
df = df[df["reviewText"].notnull()].copy()
df = df[df["reviewText"].astype(str).str.strip() != ""].copy()
df["summary"] = df["summary"].fillna("")
df["text"] = (df["summary"].astype(str) + " " + df["reviewText"].astype(str)).map(normalize_space)

# SELECT LONG REVIEWS
long_reviews = df[df["text"].apply(word_count) > MIN_WORDS_LONG_REVIEW].copy()
sample10 = long_reviews.sample(n=min(10, len(long_reviews)), random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Found {len(long_reviews)} reviews longer than 100 words.")
print(f"Selected {len(sample10)} long reviews for summarization.\n")

# LOAD MISTRAL
llm = LocalMistralGenerator(MODEL_NAME)

# SUMMARIZATION FUNCTION
def summarize_to_50_words(text: str) -> str:
    review = truncate_for_model(text)
    prompt = f"""
Summarize the following appliance review in about 50 words.
Include the main product experience, positives/negatives,
and overall customer opinion.

Review:
{review}

Summary:
"""
    output = llm.generate(prompt, max_new_tokens=90)
    return cap_to_approx_words(output, TARGET_SUMMARY_WORDS)

# GENERATE 10 SUMMARIES
print("Generating 10 summaries...\n")
summaries = []
for _, row in sample10.iterrows():
    summary = summarize_to_50_words(row["text"])
    summaries.append(summary)

# PRINT FIRST TWO FOR REPORT
print("=== FIRST TWO SUMMARIES (for report) ===")
for i in range(min(2, len(summaries))):
    print(f"\nReview #{i+1}")
    print(f"Original words: {word_count(sample10.iloc[i]['text'])}")
    print("\nOriginal Review:")
    print(sample10.iloc[i]["text"])
    print("\nSummary (~50 words):")
    print(summaries[i])

# QUESTION-LIKE REVIEW RESPONSE
q_candidates = df[df["text"].apply(is_question_like)].copy()
if q_candidates.empty:
    print("\nNo question-like review found.")
else:
    q_text = q_candidates.sample(n=1, random_state=RANDOM_STATE).iloc[0]["text"]
    q_short = truncate_for_model(q_text, max_chars=1500)
    response_prompt = f"""
You are a professional customer service representative.

Respond politely to the following customer review.
Acknowledge the issue and suggest a practical next step.

Customer Review:
{q_short}

Response:
"""
    response = llm.generate(response_prompt, max_new_tokens=120)
    print("\n=== QUESTION-LIKE REVIEW (for report) ===")
    print(q_text)
    print("\n=== GENERATED CUSTOMER SERVICE RESPONSE (for report) ===")
    print(response)

Loading dataset...
Found 49061 reviews longer than 100 words.
Selected 10 long reviews for summarization.

Loading model: mistralai/Mistral-7B-Instruct-v0.1 (this takes time...)


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Generating 10 summaries...

=== FIRST TWO SUMMARIES (for report) ===

Review #1
Original words: 168

Original Review:
good scrub but not really a peel I was hoping that this could lightly peel off my skin to help even out my skin tone but that didn't happen.. but its a really good scrub, non irritating even if you rub your face for 1-2 minutes..and you can feel that all impurities was lifted.. ive been using this for almost a month now and I use it everyday at night after I wash my face with olay facial foam cleanser.. my skin did brighten, I noticed that it lessen my whiteheads and my skin is smoother.. its really good as a scrub and I love using it.. it would have been perfect it if could lightly peel my skin off.. I think the percentage of glycolic acid is not enough.. but still, a highly recommended scrub, not a peel off :-) update: this was ok on few days of use but my skin reacted if used in a daily basis. maybe once or twice a week is ok..

Summary (~50 words):
The product exper